# Interview Scoring System in Colab

Use the same JSON configuration and `app.py` as the local notebook. Providers: `local`, `gemini`, `mistral_free`, `groq_free`, and `openrouter`. Keys stay in environment variables, selected `.key` files, or Colab Secrets, not notebook source.

Run the setup cells, edit the JSON in the repository directory, then run scoring. No model call is made during setup. The shipped configuration selects local inference; choose a cloud provider explicitly if wanted.

## Obtain the repository

While PR #1 is open, the default ref below is its feature branch. After it is merged, set `REPO_REF = 'main'`. An existing checkout is reused without resetting files or changing branches, so your edits and data are preserved.

If using a Colab local runtime, set `REPO_DIR` to your existing local checkout instead of `/content/...`. Do not put credentials in clone URLs.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/buriansandor/interview_scoring_system.git'
REPO_REF = 'feat/json-llm-provider-config'  # Use 'main' after PR #1 is merged.
REPO_DIR = Path('/content/interview_scoring_system')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Reusing existing checkout; REPO_REF is not applied to existing files.')
if not (REPO_DIR / 'notebook_support.py').is_file():
    raise RuntimeError('This checkout lacks notebook support. Select the feature branch, or main after merge, without discarding your work.')
print('Repository:', REPO_DIR)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements.txt')], check=True)


## Configure the provider and data

Edit `llm_config.json` in the checkout, or upload a private configuration and set `CONFIG_PATH` below to it. Change `provider`, its model profile, input CSV, rules JSON, and output path. Relative paths are based on the configuration file's directory.

For cloud providers, add the selected key to Colab Secrets with the configured `api_key_env` name: `GEMINI_API_KEY`, `MISTRAL_API_KEY`, `GROQ_API_KEY`, or `OPENROUTER_API_KEY`. Grant this notebook access to that secret. Only the selected missing key is requested; environment variables and `.key` files take priority. Local mode does not request Colab Secrets.

A hosted Colab runtime's `127.0.0.1` is the Colab machine, not your home computer. For a home llama.cpp/Ollama server, use the local notebook or a local runtime. Do not publicly expose an unauthenticated inference server. Free-tier account settings still apply; see `CONFIGURATION.md`.

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from notebook_support import load_notebook_config, run_notebook

CONFIG_PATH = REPO_DIR / 'llm_config.json'
config = load_notebook_config(CONFIG_PATH, use_colab_secrets=True)
print('Provider:', config.llm.provider, '| Model:', config.llm.model)
print('Output:', config.output_csv)
# Never print the secret value or store it in notebook cells.


## Score and inspect results

The shared helper runs `app.py --config ...` with the same validation and request limits as the CLI. Secrets are passed to the child process environment, not written to disk or added to command arguments. `app.py` is never rewritten.

Budget exhaustion returns saved partial results with a warning. Other errors stop the cell without loading an old CSV. Every execution starts a new run with a new application request budget; provider quotas are separate.

In [ ]:
result_df = run_notebook(CONFIG_PATH, REPO_DIR, use_colab_secrets=True)
print('Partial results:', result_df.attrs['partial_results'])
result_df.head()


## Export and privacy

The complete CSV is already saved to the configured output path. Optionally download it using the commented cell below. Avoid printing the full log or credentials; displayed interview answers may still be sensitive. Clear notebook outputs before sharing or committing.

Compare a sample with human-scored answers before research use, particularly for low-confidence rows.

In [ ]:
# Optional: uncomment to download the CSV in Colab.
# from google.colab import files
# files.download(result_df.attrs['output_csv'])
